# 04 项目报告 Notebook

这个 notebook 是精简版 P2Rp2 风格报告，也可作为最终展示图表的来源。结构对应课程要求：科学问题、数据、处理与分析、建模与解释、结论和分工说明。

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display, Image, Markdown

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src import spectral_notebook_tools as snt

ANALYSIS_DIR = PROJECT_ROOT / "output" / "analysis_pipeline"
FIG_DIR = ANALYSIS_DIR / "figures"

def read_product(filename):
    table = snt.read_combined_analysis_products(ANALYSIS_DIR, filename)
    if table.empty:
        raise FileNotFoundError(f"缺少 {filename} 或目标化 *_{filename}")
    return table

def latest_figure(filename):
    candidates = sorted(FIG_DIR.glob(f"*_{filename}"), key=lambda p: p.stat().st_mtime, reverse=True)
    legacy = FIG_DIR / filename
    if legacy.exists():
        candidates.append(legacy)
    return candidates[0] if candidates else None

target_status = read_product("target_status.csv")
line_qc = read_product("line_diagnostics_qc.csv")
host_summary = read_product("host_environment_summary.csv")
bb = read_product("blackbody_temperature.csv")

## 科学问题

对一组新观测的超新星目标，如果每个目标只有 1-4 条光学光谱，我们能可靠提取哪些光谱信息？比较稳妥的目标是：确认类型/子型，用 TNS 发现日期给出观测时间基准，在可靠时测量关键谱线速度和 pEW/FWHM，识别宿主污染或窄宿主线，并把每个目标放到公开超新星光谱样本的背景下比较。当前没有最大光日期，因此不讨论相对最大光相位。


## 数据

本项目整合 TNS 元数据和找星图、可用时的 Lasair/ZTF 光变曲线、WISeREP 公开光谱，以及 `data/SN*/` 下的本地 BFOSC 一维光谱。下面的分析产物可以由 `02_spectral_analysis_pipeline.ipynb` 逐目标调参生成，也可以由 `scripts/build_analysis_products.py` 批量生成。

In [ ]:
display(target_status)

## 处理与分析

精简 pipeline 读取已定标的一维 FITS 光谱；在有 TNS 红移时进行静止系修正；按目标类型测量合适的稀疏光谱诊断量；并给出保守质检标记。标记为 `check` 的自动测量保留在表中以便追溯，但人工看图前不应作为最终数值引用。线表中的 Ca II H&K / NIR 是 blend-proxy 测量；新版表还包含平滑/连续谱窗口扰动得到的经验 systematics 列。


In [ ]:
adopted = line_qc[line_qc["qc_flag"].eq("adopt")].copy()
display(adopted[["target", "date_obs", "phase_days", "type", "line", "velocity_kms", "pEW_A", "FWHM_A"]])

## 关键图表

下面的 code cell 会优先显示带目标名前缀的最新图；如果没有目标化图，再回退到旧的无前缀批量图。

In [ ]:
for title, fig in [
    ("目标状态", "target_status_table.png"),
    ("谱线速度演化", "line_velocity_evolution.png"),
    ("pEW 演化", "pew_evolution.png"),
    ("连续谱颜色温度估计", "blackbody_temperature.png"),
    ("宿主/环境谱线探测", "host_line_detections.png"),
]:
    path = latest_figure(fig)
    display(Markdown(f"### {title}"))
    if path is None:
        print(f"缺少图：{fig}")
    else:
        print(path.name)
        display(Image(filename=str(path)))

## 建模与解释

文献调研支持保守解释。Ia 型目标应主要用 Si II 6355 和 Ca II blend proxy 的速度和 pEW，与 BSNIP/CSP/Branch 类样本比较；Si II 5972 仅作为 visual/check context。II 型目标应优先讨论 Fe II 5169 速度量级，Balmer trough 速度需人工确认后再引用。去包层候选体应先核查 He/O/Ca 谱线识别，再讨论子型。黑体颜色温度只作为连续谱 proxy，宿主线只作为窄线指数。TARDIS 可以辅助说明谱线识别或大体谱形，但不作为前身星或爆炸参数结论的主要证据。


In [ ]:
display(host_summary)

## 当前结论

- `SN2026FVX` 和 `SN2026JLM` 具有 Ia 型稀疏光谱序列特征；Si II 6355 是第一版速度测量中最干净的谱线，Ca II 使用 blend-proxy 解释。
- `SN2026KID` 是当前最明确的 II 型案例；最终引用 Fe II 5169 和宿主线指标前仍需人工检查，Halpha/Hbeta 不自动作为 adopted 速度。
- `SN2026KIE` 应先按去包层候选体处理；子型需要结合 He/O/Ca 诊断和模板拟合进一步确认。
- `SN2026LMP` 在当前元数据中仍未分类且缺少可靠红移；确认红移和类型前，不应进入强科学结论。

这些结果已经可以作为第一版展示产物，但最终报告中的数值应基于 `adopt` 测量和人工视觉检查。所有相位均写成距发现日天数。


## 分工页/报告占位

提交前请把 `TBD by group` 替换成小组成员姓名。一个清晰分工可以是：

| 分工 | 负责人 | 贡献 |
|---|---|---|
| 观测准备 | TBD by group | TNS 元数据、找星图、观测窗口准备、公开数据盘点。 |
| 光谱处理 | TBD by group | BFOSC/FITS 检查、光谱抽取核查、波长和红移合理性检查。 |
| 光谱诊断 | TBD by group | 谱线速度、pEW/FWHM、黑体颜色温度 proxy、宿主线标记。 |
| 解释与展示 | TBD by group | 文献比较、报告 notebook、TARDIS 定性建模、最终英文 slides。 |